# Direct loss estimation (DLE)

`DirectLossEstimator` learns squared loss from labeled reference observations and compares estimated current loss with a held-out reference baseline. For regression, the mean loss is **MSE**. For binary classification with labels 0/1 and `y_pred = P(y=1)`, it is the **Brier score**.

The first two examples use feature arrays and prediction vectors directly. The final two use `DirectLossAnalyzer` to run an independent DLE per ID in a panel. `degradation` tests whether **estimated** mean loss increased beyond a chosen relative margin.

In [1]:
import os
import sys

sys.path.append(os.path.abspath("../.."))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

from tinyshift.performance import DirectLossAnalyzer, DirectLossEstimator

rng = np.random.default_rng(42)

## 1. DLE for regression vectors

The monitored model predicts a numeric target. Its squared error grows with `risk`, and the current batch contains more high-risk observations. The DLE fits its learner on the first 75% of reference rows, uses the last 25% as a baseline, and returns a structured comparison for the unlabeled current batch. At least two rows are required in the fitting, held-out, and current groups.

In [2]:
reference_risk = rng.uniform(0, 1, 400)
current_risk = rng.beta(8, 2, 120)
X_reference = reference_risk.reshape(-1, 1)
X_current = current_risk.reshape(-1, 1)
predictions_reference = 10 + 0.5 * reference_risk
predictions_current = 10 + 0.5 * current_risk
y_reference = predictions_reference + 0.3 + 1.5 * reference_risk + rng.normal(0, 0.04, 400)

regression_dle = DirectLossEstimator(
    learner=RandomForestRegressor(n_estimators=80, min_samples_leaf=5, random_state=42),
    fraction=0.25,
    alpha=0.05,
    n_resamples=999,
    random_state=42,
).fit(X_reference, y_reference, predictions_reference)
regression_result = regression_dle.predict(
    X_current, predictions_current, degradation_margin=0.10
)
regression_result

DirectLossResult(reference_estimated=1.344287354731429, reference_realized=1.3335660140053947, reference_size=100, current_estimated=2.318434235768232, estimated_delta=0.974146881036803, relative_delta=0.7246567317680563, degradation_margin=0.1, p_value=0.001, degradation=True, current_size=120)

`predict(..., degradation_margin=0.10)` tests whether the current mean estimated loss rose by more than 10% relative to the held-out reference. `DirectLossResult` reports the absolute change (`estimated_delta`), relative change (`relative_delta`), tested margin, one-sided `p_value`, and `degradation`. The test divides current per-row losses by `1 + margin` and permutes them with the reference losses using Welch's t statistic. An alert requires both `relative_delta > degradation_margin` and `p_value <= alpha`.

For the numeric mean estimate or per-row estimates, use `estimate` or `estimate_loss` respectively:

In [3]:
regression_dle.estimate(X_current, predictions_current), regression_dle.estimate_loss(X_current, predictions_current)[:5]

(2.318434235768232, array([2.31842971, 1.86047363, 2.77530705, 2.26997538, 1.86084738]))

The estimated increase above is about 72%. Raising the margin to 80% asks a stricter question and produces no alert for the same batch. A `False` decision does not establish that performance is unchanged.

In [4]:
regression_dle.predict(
    X_current, predictions_current, degradation_margin=0.80
)

DirectLossResult(reference_estimated=1.344287354731429, reference_realized=1.3335660140053947, reference_size=100, current_estimated=2.318434235768232, estimated_delta=0.974146881036803, relative_delta=0.7246567317680563, degradation_margin=0.8, p_value=0.735, degradation=False, current_size=120)

## 2. DLE for binary probability vectors

Use `P(y=1)` as the prediction vector and numeric labels 0/1. The current probabilities are closer to 0.5, where a calibrated classifier generally has higher expected Brier loss. The result uses the same fields as in regression.

In [5]:
binary_rng = np.random.default_rng(18)
p_reference = binary_rng.beta(1.5, 1.5, 600)
p_current = binary_rng.beta(10, 10, 200)
y_binary_reference = binary_rng.binomial(1, p_reference)
X_binary_reference = np.zeros((len(p_reference), 1))
X_binary_current = np.zeros((len(p_current), 1))

brier_dle = DirectLossEstimator(
    learner=RandomForestRegressor(n_estimators=80, min_samples_leaf=10, random_state=42),
    fraction=0.25,
    random_state=42,
).fit(X_binary_reference, y_binary_reference, p_reference)
brier_dle.predict(X_binary_current, p_current, degradation_margin=0.20)

DirectLossResult(reference_estimated=0.17660432453501215, reference_realized=0.21007464743363766, reference_size=150, current_estimated=0.23148659130542512, estimated_delta=0.05488226677041297, relative_delta=0.31076400260817194, degradation_margin=0.2, p_value=0.005, degradation=True, current_size=200)

## 3. Analyzer for regression panels

`DirectLossAnalyzer` clones one DLE per ID and collects its `DirectLossResult` into a DataFrame. Each DLE performs its own reference split and comparison. Preserve the intended row order; for a time series, order rows chronologically within each ID before fitting.

In [ ]:
def regression_batch(store, risk, labeled):
    prediction = 10 + 0.5 * risk
    frame = pd.DataFrame({"unique_id": store, "risk": risk, "y_pred": prediction})
    if labeled:
        error = 0.3 + 1.5 * risk if store == "store_A" else 0.6 + 0.5 * risk
        frame["y"] = prediction + error + rng.normal(0, 0.04, len(risk))
    return frame


regression_reference = pd.concat([
    regression_batch("store_A", rng.uniform(0, 1, 400), labeled=True),
    regression_batch("store_B", rng.uniform(0, 1, 400), labeled=True),
], ignore_index=True)
regression_current = pd.concat([
    regression_batch("store_A", rng.beta(8, 2, 120), labeled=False),
    regression_batch("store_B", rng.beta(2, 8, 120), labeled=False),
], ignore_index=True)

regression_analyzer = DirectLossAnalyzer(
    estimator=DirectLossEstimator(
        learner=RandomForestRegressor(n_estimators=80, min_samples_leaf=5, random_state=42),
        random_state=42,
    ),
    fraction=0.25,
).fit(regression_reference, feature_cols=["risk"])
regression_analyzer.predict(regression_current, degradation_margin=0.10)

,unique_id,reference_estimated,reference_realized,reference_size,current_estimated,estimated_delta,relative_delta,degradation_margin,p_value,degradation,current_size
0,store_A,1.301065,1.321112,100,2.246220,0.945154,0.726446,0.1,0.001,True,120
1,store_B,0.769801,0.772781,100,0.503652,-0.266149,-0.345738,0.1,1.000,False,120


`reference_realized` is observed MSE on held-out reference rows. `reference_estimated` and `current_estimated` come from the learned loss model. `estimated_delta` and `relative_delta` compare those estimates. The one-sided `p_value` tests whether the increase exceeds the chosen margin; the analyzer’s `results_` stores one `DirectLossResult` per ID. This test concerns estimated loss, not confirmed realized performance.

## 4. Analyzer for binary probability panels

The same analyzer accepts one `y_pred` probability column and 0/1 reference labels. Here model A becomes less confident and model B more confident in the current batch. The output loss is Brier score, because it averages `(y - P(y=1))²`.

In [ ]:
panel_rng = np.random.default_rng(29)


def binary_batch(model_id, probabilities, labeled):
    frame = pd.DataFrame({
        "unique_id": model_id,
        "segment": np.zeros(len(probabilities)),
        "y_pred": probabilities,
    })
    if labeled:
        frame["y"] = panel_rng.binomial(1, probabilities)
    return frame


binary_reference = pd.concat([
    binary_batch("model_A", panel_rng.beta(1.5, 1.5, 600), labeled=True),
    binary_batch("model_B", panel_rng.beta(1.5, 1.5, 600), labeled=True),
], ignore_index=True)
binary_current = pd.concat([
    binary_batch("model_A", panel_rng.beta(10, 10, 200), labeled=False),
    binary_batch("model_B", panel_rng.beta(0.5, 0.5, 200), labeled=False),
], ignore_index=True)

brier_analyzer = DirectLossAnalyzer(
    estimator=DirectLossEstimator(
        learner=RandomForestRegressor(n_estimators=80, min_samples_leaf=10, random_state=42),
        random_state=42,
    ),
    fraction=0.25,
).fit(binary_reference, feature_cols=["segment"])
brier_analyzer.predict(binary_current, degradation_margin=0.10)

,unique_id,reference_estimated,reference_realized,reference_size,current_estimated,estimated_delta,relative_delta,degradation_margin,p_value,degradation,current_size
0,model_A,0.191231,0.188216,150,0.246546,0.055316,0.289262,0.1,0.001,True,200
1,model_B,0.170662,0.179364,150,0.126863,-0.043800,-0.256645,0.1,1.000,False,200


Current labels are absent in both panels, so actual current loss cannot yet be checked. Compare estimated and realized loss when labels arrive. Estimates may become inaccurate if the relationship between inputs and loss changes; for classification, changed probability calibration is one such case. Each current ID needs at least two rows. Row-wise permutation assumes approximately independent observations; temporal dependence can make the p-value unreliable.